# L10c: Autoencoders: Learning Compressed Representations
In L9a through L10a, we learned word embeddings: dense, low-dimensional vectors that capture semantic relationships between words. Each method (BoW, CBOW, Skip-Gram, GloVe) learns a compressed representation of text by exploiting co-occurrence structure in a corpus. Autoencoders generalize this idea beyond text. An autoencoder is a neural network trained to reconstruct its input through a bottleneck, forcing the network to learn a compressed representation of the data. The bottleneck activations are the embedding — a low-dimensional code that captures the structure needed to reconstruct the original input.

This lecture introduces the autoencoder architecture, connects linear autoencoders to PCA (from L3a), and describes how nonlinear activations enable autoencoders to learn representations on curved manifolds that PCA cannot capture.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Autoencoder architecture and reconstruction loss:__ Describe the encoder-bottleneck-decoder structure and the mean squared error reconstruction loss. Explain how the bottleneck forces the network to learn a compressed representation of the input.
> * __Linear autoencoders and PCA:__ Show that a single-layer autoencoder with linear activations learns the same subspace as PCA. Explain why nonlinear activations are needed to go beyond PCA.
> * __Regularized autoencoders:__ Describe denoising and sparse autoencoders and explain how each regularization strategy prevents the network from learning the identity function.

Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Train an Autoencoder on MNIST](CHEME-5820-L10c-Example-Autoencoder-Spring-2026.ipynb). In this example, we train a nonlinear autoencoder on MNIST handwritten digits, visualize the learned bottleneck representations in 2D, and compare the reconstruction quality and embedding structure to PCA.

___

<div>
  <center>
    <img src="figs/Fig-Autoencoder-Architecture.svg" style="width:80%" />
  </center>
</div>

## Autoencoder Architecture
An autoencoder is a feedforward neural network (L8c) trained to map its input to itself through a bottleneck layer that has fewer nodes than the input. The network consists of two parts: an encoder that compresses the input into a low-dimensional code, and a decoder that reconstructs the input from that code.

> __Definition (Autoencoder)__
>
> An autoencoder consists of an encoder $f_{\mathrm{enc}}(\cdot;\theta_{\mathrm{enc}}):\mathbb{R}^{d_{\mathrm{in}}}\rightarrow\mathbb{R}^{d}$ and a decoder $f_{\mathrm{dec}}(\cdot;\theta_{\mathrm{dec}}):\mathbb{R}^{d}\rightarrow\mathbb{R}^{d_{\mathrm{in}}}$, where $d < d_{\mathrm{in}}$ is the bottleneck dimension and $\theta = (\theta_{\mathrm{enc}}, \theta_{\mathrm{dec}})$ collects all learnable parameters (weights and biases). Given input $\mathbf{x}\in\mathbb{R}^{d_{\mathrm{in}}}$, the encoder maps the input to a low-dimensional code $\mathbf{z} = f_{\mathrm{enc}}(\mathbf{x};\theta_{\mathrm{enc}}) \in \mathbb{R}^{d}$, and the decoder reconstructs the input from that code as $\hat{\mathbf{x}} = f_{\mathrm{dec}}(\mathbf{z};\theta_{\mathrm{dec}}) \in \mathbb{R}^{d_{\mathrm{in}}}$. The full autoencoder computes $\hat{\mathbf{x}} = f_{\mathrm{dec}}(f_{\mathrm{enc}}(\mathbf{x};\theta_{\mathrm{enc}});\theta_{\mathrm{dec}})$, and the code $\mathbf{z}$ is the learned embedding of $\mathbf{x}$.

The encoder and decoder are each composed of one or more fully connected layers with activation functions, following the same function composition structure as the feedforward networks in L8c. The bottleneck constraint $d < d_{\mathrm{in}}$ is what forces the network to learn a useful representation: it cannot simply copy the input to the output, so it must discover which features of the input are most important for reconstruction.

### Reconstruction Loss
The autoencoder is trained to minimize the difference between the input and its reconstruction. For a dataset of $N$ examples $\{\mathbf{x}^{(1)}, \dots, \mathbf{x}^{(N)}\}$, the loss function is:

> __Definition (Reconstruction loss)__
>
> The mean squared error (MSE) reconstruction loss over $N$ training examples is:
> $$
> \mathcal{L}(\theta) = \frac{1}{N}\sum_{n=1}^{N} \|\mathbf{x}^{(n)} - \hat{\mathbf{x}}^{(n)}(\theta)\|_{2}^{2}
> $$
> where $\hat{\mathbf{x}}^{(n)}(\theta) = f_{\mathrm{dec}}(f_{\mathrm{enc}}(\mathbf{x}^{(n)};\theta_{\mathrm{enc}});\theta_{\mathrm{dec}})$ is the reconstruction of input $\mathbf{x}^{(n)}$.
> 
> The parameters $\theta$ are optimized using backpropagation and gradient descent (L8c), the same training procedure used for any feedforward network. The only difference is the training target: instead of predicting a label $\mathbf{y}$, the autoencoder predicts the input $\mathbf{x}$ itself. For binary or normalized inputs (e.g., pixel values in $[0,1]$), binary cross-entropy is sometimes used instead of MSE:
>$$
\mathcal{L}_{\mathrm{BCE}}(\theta) = -\frac{1}{N}\sum_{n=1}^{N}\sum_{i=1}^{d_{\mathrm{in}}} \left[\underbrace{x_{i}^{(n)}\log\hat{x}_{i}^{(n)}(\theta)}_{\text{class} = 1} + \underbrace{(1-x_{i}^{(n)})\log(1-\hat{x}_{i}^{(n)}(\theta))}_{\text{class} = 0}\right]
>$$
> where the decoder output layer uses a sigmoid activation so that $\hat{x}_{i}^{(n)}(\theta)\in(0,1)$.

We solve this optimization problem using the Adam optimizer, which we introduced in L9c and L10a as a comparison to AdaGrad. If you want to understand how Adam works in detail, check out the advanced notebook on Adam:

> [▶ The Adam Optimizer](CHEME-5820-L10c-Advanced-Adam-Spring-2026.ipynb). This advanced notebook examines the Adam optimizer used to train the autoencoder: the motivation behind its design, the full update rule with bias correction, and practical guidance for hyperparameter selection.

___

## Linear Autoencoders and PCA
What happens when the encoder and decoder use no activation functions (i.e., all activations are the identity)? The autoencoder reduces to a linear map, and the optimal solution recovers the same subspace as Principal Component Analysis (PCA)!

> __Definition (Linear autoencoder)__
>
> A linear autoencoder with a single encoder layer and a single decoder layer takes input $\mathbf{x}\in\mathbb{R}^{d_{\mathrm{in}}}$ and computes a code $\mathbf{z} = \mathbf{W}_{\mathrm{enc}}\,\mathbf{x}\in\mathbb{R}^{d}$ and a reconstructed input $\hat{\mathbf{x}} = \mathbf{W}_{\mathrm{dec}}\,\mathbf{z}\in\mathbb{R}^{d_{\mathrm{in}}}$, where $\mathbf{W}_{\mathrm{enc}}\in\mathbb{R}^{d\times d_{\mathrm{in}}}$ is the encoder weight matrix and $\mathbf{W}_{\mathrm{dec}}\in\mathbb{R}^{d_{\mathrm{in}}\times d}$ is the decoder weight matrix. Because there are no activation functions, the full reconstruction $\hat{\mathbf{x}} = \mathbf{W}_{\mathrm{dec}}\mathbf{W}_{\mathrm{enc}}\,\mathbf{x}$ is a rank-$d$ linear projection of the input $\mathbf{x}$. The learnable parameters are $\theta = (\mathbf{W}_{\mathrm{enc}}, \mathbf{W}_{\mathrm{dec}})$.

Minimizing $\|\mathbf{x} - \mathbf{W}_{\mathrm{dec}}\mathbf{W}_{\mathrm{enc}}\mathbf{x}\|_{2}^{2}$ over all training examples is equivalent to finding the rank-$d$ matrix that best approximates the data in the least-squares sense. From L3a, we know that the optimal rank-$d$ approximation of a data matrix is given by its truncated SVD (Eckart-Young theorem), which spans the same subspace as the top $d$ principal components.

> __Connection to PCA__
>
> In L3a, we learned that PCA finds the $d$ directions in the data that capture the most variance. These are the top $d$ right singular vectors $\boldsymbol{\phi}_{1},\dots,\boldsymbol{\phi}_{d}$ from the SVD of the centered data matrix $\mathbf{A} = \mathbf{U}\mathbf{S}\mathbf{V}^{\top}$. PCA compresses each input by projecting it onto these $d$ directions, and reconstructs by mapping back.
>
> A linear autoencoder, trained to minimize reconstruction error, discovers the same $d$ directions. The decoder weight matrix $\mathbf{W}_{\mathrm{dec}}$ learns columns that span the same subspace as the top $d$ principal components:
> $$
> \mathrm{colspan}(\mathbf{W}_{\mathrm{dec}}) = \mathrm{span}(\boldsymbol{\phi}_{1},\dots,\boldsymbol{\phi}_{d})
> $$
> The encoder projects each input onto these directions to produce the code $\mathbf{z}$, and the decoder reconstructs from them. Because this is the best rank-$d$ approximation of the data (by the Eckart-Young theorem, L3a), the linear autoencoder cannot do better than PCA, it converges to the same solution.
>
> __Reference:__ [Baldi, P. & Hornik, K. (1989). Neural Networks and Principal Component Analysis: Learning from Examples Without Local Minima. Neural Networks, 2(1), 53-58.](https://doi.org/10.1016/0893-6080(89)90014-2)

This result means that a linear autoencoder cannot learn anything that PCA does not already capture. To go beyond PCA, we need nonlinear activations.
___

## Nonlinear Autoencoders
Adding nonlinear activation functions (ReLU, sigmoid, tanh) to the encoder and decoder layers allows the autoencoder to learn representations on curved manifolds that linear methods cannot capture.

> __Definition (Nonlinear autoencoder)__
>
> A nonlinear autoencoder with one hidden layer in each of the encoder and decoder computes:
> $$
> \begin{align*}
> \mathbf{z} &= \sigma_{\mathrm{enc}}(\mathbf{W}_{\mathrm{enc}}\,\mathbf{x} + \mathbf{b}_{\mathrm{enc}}) \in \mathbb{R}^{d} \\
> \hat{\mathbf{x}} &= \sigma_{\mathrm{dec}}(\mathbf{W}_{\mathrm{dec}}\,\mathbf{z} + \mathbf{b}_{\mathrm{dec}}) \in \mathbb{R}^{d_{\mathrm{in}}}
> \end{align*}
> $$
> where $\sigma_{\mathrm{enc}}$ and $\sigma_{\mathrm{dec}}$ are element-wise nonlinear activation functions, and $\mathbf{b}_{\mathrm{enc}}, \mathbf{b}_{\mathrm{dec}}$ are bias vectors. The learnable parameters $\theta = (\mathbf{W}_{\mathrm{enc}}, \mathbf{b}_{\mathrm{enc}}, \mathbf{W}_{\mathrm{dec}}, \mathbf{b}_{\mathrm{dec}})$ are trained by minimizing the same reconstruction loss $\mathcal{L}(\theta)$ defined above using backpropagation and gradient descent. Deeper autoencoders stack multiple layers in the encoder and decoder, following the function composition from L8c.

__Why does nonlinearity matter?__ A linear autoencoder can only do what PCA already does. Nonlinear activations let the encoder bend and reshape the input space before compressing it, so the bottleneck code $\mathbf{z}$ can preserve structure that PCA would destroy.

### Architecture Choices
The architecture of an autoencoder involves several design decisions.

> __Design considerations__
>
> * __Bottleneck dimension $d$:__ Controls the compression ratio. Smaller $d$ forces more aggressive compression but may lose important features. The choice depends on the intrinsic dimensionality of the data.
> * __Depth:__ Deeper encoders and decoders can learn more complex transformations but require more data and are harder to train. A common pattern is a symmetric architecture where the decoder mirrors the encoder layer sizes.
> * __Activation functions:__ ReLU is standard for hidden layers. The output activation depends on the data: sigmoid for inputs in $[0,1]$, linear for unbounded inputs.
> * __Tied weights:__ Setting $\mathbf{W}_{\mathrm{dec}} = \mathbf{W}_{\mathrm{enc}}^{\top}$ halves the number of free parameters the model must learn, reducing model capacity and making it harder to memorize training data.

Without regularization, a sufficiently large autoencoder can learn the identity function, memorizing each training example without learning generalizable structure. The next section describes two strategies to prevent this.
___

## Regularized Autoencoders
If the autoencoder has enough capacity, it can achieve zero reconstruction loss without learning anything useful. 

__Consider an extreme case:__ suppose the bottleneck dimension equals the input dimension ($d = d_{\mathrm{in}}$) and the activation functions are the identity. Then the encoder and decoder can each learn the identity matrix, so $\hat{\mathbf{x}} = \mathbf{x}$ for every input. The reconstruction is perfect, but the bottleneck code $\mathbf{z} = \mathbf{x}$ is just a copy of the input, with no compression at all. 

Even with $d < d_{\mathrm{in}}$, a sufficiently deep nonlinear network can memorize individual training examples rather than learning shared structure across them. Regularization prevents this by adding constraints that force the encoder to learn structure rather than memorize data.

### Denoising Autoencoders
Real-world data is noisy. Sensor readings have measurement error, images have pixel noise, and experimental measurements have instrument variability. A useful representation should capture the underlying signal, not the noise in a particular observation. A denoising autoencoder builds this directly into training: it corrupts the input on purpose and forces the network to recover the clean version.

> __Definition (Denoising autoencoder)__
>
> Given input $\mathbf{x}$, generate a corrupted version $\tilde{\mathbf{x}}$ by applying a noise process (e.g., adding Gaussian noise $\tilde{\mathbf{x}} = \mathbf{x} + \boldsymbol{\epsilon}$ where $\boldsymbol{\epsilon}\sim\mathcal{N}(\mathbf{0}, \sigma^{2}\mathbf{I})$, or randomly zeroing out a fraction of input components). The denoising autoencoder minimizes:
> $$
> \mathcal{L}_{\mathrm{DAE}}(\theta) = \frac{1}{N}\sum_{n=1}^{N} \|\mathbf{x}^{(n)} - f_{\mathrm{dec}}(f_{\mathrm{enc}}(\tilde{\mathbf{x}}^{(n)}))\|_{2}^{2}
> $$
> The network receives $\tilde{\mathbf{x}}$ but must reconstruct $\mathbf{x}$, so it cannot simply copy its input.
>
> __Reference:__ [Vincent, P., et al. (2008). Extracting and Composing Robust Features with Denoising Autoencoders. ICML 2008.](https://dl.acm.org/doi/10.1145/1390156.1390294)

Because the corrupted input differs from the original on every forward pass, the identity mapping fails. The encoder must learn to extract the underlying signal from noisy observations, producing representations that are robust to input perturbations.

> __Connection to Hopfield networks (L6a):__ This task, recovering a clean pattern from a corrupted version, is the same problem that classical Hopfield networks solve. In L6a, we presented a partial or noisy pattern and let the network dynamics converge to the nearest stored memory. A denoising autoencoder does the same thing in a single feedforward pass: it maps a corrupted input directly to the clean reconstruction, with the mapping learned by gradient descent rather than defined by Hebbian weights and energy minimization.

### Sparse Autoencoders
A sparse autoencoder adds a penalty that encourages most bottleneck units to be inactive for any given input.

> __Definition (Sparse autoencoder)__
>
> A sparse autoencoder adds an $\ell_1$ penalty on the bottleneck activations to the reconstruction loss:
> $$
> \mathcal{L}_{\mathrm{SAE}}(\theta) = \frac{1}{N}\sum_{n=1}^{N} \|\mathbf{x}^{(n)} - \hat{\mathbf{x}}^{(n)}\|_{2}^{2} + \lambda\sum_{n=1}^{N}\|\mathbf{z}^{(n)}\|_{1}
> $$
> where $\lambda > 0$ controls the sparsity-reconstruction trade-off and $\|\mathbf{z}^{(n)}\|_{1} = \sum_{j=1}^{d}|z_{j}^{(n)}|$. Larger $\lambda$ produces sparser codes.

Sparsity encourages each input to be represented by a small number of active bottleneck units, producing codes where different units correspond to different features of the data. Unlike the standard bottleneck autoencoder, a sparse autoencoder can have $d \geq d_{\mathrm{in}}$ (an overcomplete representation) without learning the identity, because the sparsity penalty prevents all units from being active simultaneously.

___

## Autoencoders as Embedding Methods
After training, the decoder is no longer needed. The encoder alone maps any input $\mathbf{x}\in\mathbb{R}^{d_{\mathrm{in}}}$ to a bottleneck code $\mathbf{z}\in\mathbb{R}^{d}$ that captures the features most important for reconstruction. This code is the embedding, a dense low-dimensional representation of the input that can be used for downstream tasks such as classification, clustering, or retrieval, or visualized directly when $d = 2$ or $d = 3$.

This is the same idea behind the word embedding methods from L9a through L10a, but autoencoders are not limited to text. They work with any data type (images, spectra, molecular descriptors, time series) because the training signal is reconstruction, not co-occurrence.

> __Embeddings across methods__
>
> * __CBOW / Skip-Gram (L9a, L9c):__ The embedding is a column of the input weight matrix $\mathbf{W}_{1}$, learned as a byproduct of predicting context or target words. The training signal comes from local co-occurrence structure in a text corpus, so these embeddings are specific to language data.
> * __GloVe (L10a):__ The embedding is the sum $\mathbf{w}_{i} + \tilde{\mathbf{w}}_{i}$, learned by factorizing the log co-occurrence matrix. Like CBOW and Skip-Gram, GloVe exploits word co-occurrence statistics, but it operates on global counts rather than local context windows.
> * __Autoencoder:__ The embedding is the bottleneck activation $\mathbf{z} = f_{\mathrm{enc}}(\mathbf{x};\theta_{\mathrm{enc}})$, learned by reconstructing the input through a compressed code. Because the training signal is reconstruction rather than co-occurrence, autoencoders can embed any data type where reconstruction loss is meaningful.

> [▶ Train an Autoencoder on MNIST](CHEME-5820-L10c-Example-Autoencoder-Spring-2026.ipynb). In this example, we train a nonlinear autoencoder on MNIST handwritten digits, visualize the learned bottleneck representations in 2D, and compare the reconstruction quality and embedding structure to PCA.

___

## Lab
In the lab, we will train autoencoders on the Fun faces dataset, explore how different bottleneck dimensions and architectures affect reconstruction quality, and compare the learned representations to PCA.

## Summary
Autoencoders learn compressed representations by training a neural network to reconstruct its input through a low-dimensional bottleneck, generalizing the embedding concept from text-specific methods to arbitrary data types.

> __Key Takeaways__:
>
> * __Bottleneck forces compression:__ The encoder-bottleneck-decoder architecture forces the network to learn which features of the input are most important for reconstruction. The bottleneck activations $\mathbf{z} = f_{\mathrm{enc}}(\mathbf{x})$ serve as a learned embedding of the input.
> * __Linear autoencoders recover PCA:__ A single-layer autoencoder with linear activations learns the same subspace as the top $d$ principal components. Nonlinear activations allow autoencoders to capture curved manifold structure that PCA cannot represent.
> * __Regularization prevents memorization:__ Denoising autoencoders corrupt the input to force the encoder to learn robust features. Sparse autoencoders penalize bottleneck activations to produce codes where different units correspond to different data features.

Autoencoders provide a general framework for learning embeddings from any data type, complementing the text-specific methods from L9a through L10a.

___

Sources for this lecture:
* [Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning, Chapter 14: Autoencoders.](https://www.deeplearningbook.org/contents/autoencoders.html)
* [Baldi, P. & Hornik, K. (1989). Neural Networks and Principal Component Analysis: Learning from Examples Without Local Minima. Neural Networks, 2(1), 53-58.](https://doi.org/10.1016/0893-6080(89)90014-2)
* [Vincent, P., Larochelle, H., Bengio, Y., & Manzagol, P.-A. (2008). Extracting and Composing Robust Features with Denoising Autoencoders. ICML 2008.](https://dl.acm.org/doi/10.1145/1390156.1390294)

___